In [ ]:
import torch
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.optimize import linear_sum_assignment
from PhiKmeans import PhiKMeans

# Data
def load_iris_data():
    data = load_iris()
    return data.data, data.target

def load_wine_data():
    data = load_wine()
    return data.data, data.target

def load_cancer_data():
    data = load_breast_cancer()
    return data.data, data.target

datasets = {
    "iris": load_iris_data,
    "wine": load_wine_data,
    "cancer": load_cancer_data,
}

# Align labels
def align_labels(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    D = max(y_pred.max(), y_true.max()) + 1
    cost = np.zeros((D, D))
    for i in range(len(y_true)):
        cost[y_pred[i], y_true[i]] += 1
    row, col = linear_sum_assignment(-cost)
    mapping = {r: c for r, c in zip(row, col)}
    return np.array([mapping[l] for l in y_pred])



In [3]:
# XP
device = "cuda" if torch.cuda.is_available() else "cpu"
alphas = np.arange(0.1, 2.0, 0.1)

# Results
results = []

for name, loader in datasets.items():
    print(f"\nDataset: {name}")
    X, y = loader()
    y = LabelEncoder().fit_transform(y)
    X = StandardScaler().fit_transform(X)
    X = torch.tensor(X, dtype=torch.float32)
    k = len(np.unique(y))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # same init for all alpha
        init_centroids = PhiKMeans.kmeans_pp_init(X_train, k)

        best_alpha = None
        best_f1 = -1
        best_model = None

        for alpha in alphas:
            model = PhiKMeans(k, alpha=alpha, device=device)
            model.fit(X_train, init_centroids)

            pred = model.labels_.cpu().numpy()
            pred = align_labels(y_train, pred)

            f1 = f1_score(y_train, pred, average="macro")

            if f1 > best_f1:
                best_f1 = f1
                best_alpha = alpha
                best_model = model

        # test evaluation
        test_pred = best_model.predict(X_test).cpu().numpy()
        test_pred = align_labels(y_test, test_pred)

        precision = precision_score(y_test, test_pred, average="macro")
        recall = recall_score(y_test, test_pred, average="macro")
        f1 = f1_score(y_test, test_pred, average="macro")

        results.append({
            "dataset": name,
            "fold": fold,
            "alpha": best_alpha,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

        print(f"Fold {fold} | alpha={best_alpha:.2f} | F1={f1:.4f}")

df_results = pd.DataFrame(results)
df_results
df_results.groupby("dataset")[["precision","recall","f1"]].mean()


Dataset: iris
Fold 0 | alpha=1.00 | F1=0.8778
Fold 1 | alpha=1.80 | F1=0.8308
Fold 2 | alpha=1.40 | F1=0.8381

Dataset: wine
Fold 0 | alpha=1.00 | F1=0.9500
Fold 1 | alpha=0.70 | F1=0.9690
Fold 2 | alpha=1.60 | F1=0.9172

Dataset: cancer
Fold 0 | alpha=0.30 | F1=0.9398
Fold 1 | alpha=0.50 | F1=0.8955
Fold 2 | alpha=0.50 | F1=0.9098

Dataset: digits
Fold 0 | alpha=0.50 | F1=0.6599
Fold 1 | alpha=0.50 | F1=0.7108
Fold 2 | alpha=0.60 | F1=0.6564


,precision,recall,f1
dataset,,,
cancer,0.917821,0.916856,0.915038
digits,0.685949,0.689392,0.675678
iris,0.882721,0.855801,0.848881
wine,0.944765,0.953301,0.945419
